# NB25 — Split-Magnitude Permutation Test

**Exploratory post-hoc sensitivity** — pre-registered in METHODS.md §18 as exploratory.

**Purpose:** Test whether the observed Δβ ≈ 0.036 (resistance β ≈ 0 vs cofactor β = −0.033)
is unusual relative to 1,000 random coreness-matched partitions of the full 730-KO metal-gene
list into groups of equal size (n=106 and n=7).

**Design:**
1. One Spark query → per-genus per-KO presence matrix (cached locally)
2. 1,000 random partitions: sample 106 KOs (group A) and 7 KOs (group B) from the 730-KO list,
   stratified by coreness decile to match the actual resistance/cofactor distributions
3. For each partition: per-genus density → PGLS → Δβ = β_A − β_B
4. Empirical p = fraction of null Δβ ≥ observed Δβ
5. Same test for comparison families (AMR, TCS, ABC) from NB19

**Label:** Exploratory (designed post-H1-confirmation). Cannot shift H1 classification.

In [1]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd

DATA  = Path('../data')
FIGS  = Path('../figures')
sys.path.insert(0, str(Path('../scripts')))
from pgls_utils import run_pgls
import dendropy

SEED    = 42
N_PERM  = 1000
TREE_PATH = DATA / 'gtdb_bac_genus_pruned.tree'

## Block 0 — Load gene list, coreness, bac_base

In [2]:
gene_df = pd.read_csv(DATA / 'curated_mrg_ko_ids_v2.csv')
print('Full gene list:', len(gene_df), 'KOs')
print(gene_df['primary_category'].value_counts())

resist_kos   = set(gene_df[gene_df['primary_category'] == 'Resistance/Detoxification']['KO'])
cofactor_kos = set(gene_df[gene_df['primary_category'] == 'Cofactor Biosynthesis']['KO'])
all_kos      = list(gene_df['KO'].unique())
print('\nResistance KOs:', len(resist_kos))
print('Cofactor KOs:  ', len(cofactor_kos))

coreness_df = pd.read_csv(DATA / 'ko_coreness_pangenome.csv')
ko_coreness_map = dict(zip(coreness_df['ko'], coreness_df['coreness']))
gene_df['coreness'] = gene_df['KO'].map(ko_coreness_map)
gene_df['coreness_decile'] = pd.qcut(
    gene_df['coreness'].fillna(gene_df['coreness'].median()),
    q=10, labels=False, duplicates='drop'
)

bac_base = pd.read_csv(DATA / '01_pgls_input_bacteria.csv')
print('\nbac_base genera:', len(bac_base))

Full gene list: 730 KOs
primary_category
Unknown                       302
Transport/Homeostasis         213
Resistance/Detoxification     106
Metal-dependent Metabolism     54
Sensing/Regulation             48
Cofactor Biosynthesis           7
Name: count, dtype: int64

Resistance KOs: 106
Cofactor KOs:   7

bac_base genera: 1574


## Block 1 — Spark: per-genus per-KO presence matrix

Run once in JupyterHub. Results cached to `data/nb25_ko_presence_matrix.parquet`.

**Note on taxonomy join:** The query below assumes `kbase.ke_pangenome.genome_metadata`
has a `genus` column. If this table or column does not exist, replace with the correct
table (e.g., join via `ncbi_biosample_accession_id → nmdc.ncbi_biosamples → genus`).
Verify schema before running.

In [3]:
CACHE = DATA / 'nb25_ko_presence_matrix.parquet'

if CACHE.exists():
    presence_df = pd.read_parquet(CACHE)
    print('Loaded from cache:', presence_df.shape)
else:
    try:
        from berdl_utils import get_spark_session
        spark = get_spark_session()
    except ImportError:
        from pyspark.sql import SparkSession
        spark = SparkSession.builder.appName('nb25_presence').getOrCreate()

    quoted_kos = ','.join(f"'{k}'" for k in all_kos)

    sql = f"""
        WITH gene_ko AS (
            SELECT g.genome_id,
                   REGEXP_EXTRACT(TRIM(ko_item), '(K[0-9]{{5}})', 1) AS ko
            FROM kbase.ke_pangenome.gene g
            JOIN kbase.ke_pangenome.bakta_annotations ann
              ON g.gene_id = ann.gene_cluster_id
            LATERAL VIEW EXPLODE(SPLIT(ann.kegg_orthology_id, ',')) t AS ko_item
            WHERE ann.kegg_orthology_id IS NOT NULL
              AND TRIM(ann.kegg_orthology_id) != ''
        ),
        gene_ko_filtered AS (
            SELECT genome_id, ko FROM gene_ko
            WHERE ko IN ({quoted_kos})
              AND ko IS NOT NULL AND ko != ''
        ),
        genome_tax AS (
            SELECT 
                genome_id,
                LOWER(SPLIT(gtdb_taxonomy_id, ';')[5]) AS genus_lower
            FROM kbase.ke_pangenome.genome
            WHERE gtdb_taxonomy_id IS NOT NULL
        )
        SELECT
            gt.genus_lower,
            gkf.ko,
            COUNT(DISTINCT gkf.genome_id) AS n_genomes_with_ko
        FROM gene_ko_filtered gkf
        JOIN genome_tax gt ON gkf.genome_id = gt.genome_id
        WHERE gt.genus_lower IS NOT NULL
        GROUP BY gt.genus_lower, gkf.ko
    """

    presence_df = spark.sql(sql).toPandas()
    presence_df.attrs = {}   # <-- Fix: clear metadata
    presence_df.to_parquet(CACHE)
    print('Computed and cached:', presence_df.shape)

Loaded from cache: (417298, 3)


## Block 2 — Build genus × KO presence matrix locally

In [4]:
# Strip GTDB g__ prefix that comes from SPLIT(gtdb_taxonomy_id, ';')[5]
# bac_base uses plain genus names (e.g. 'lactobacillus'), cache has 'g__lactobacillus'
presence_df['genus_lower'] = (
    presence_df['genus_lower']
    .str.replace(r'^g__', '', regex=True)
    .str.strip()
)

# Filter to bac_base genera and pivot to wide format
bac_genera = set(bac_base['genus_lower'])
piv = presence_df[presence_df['genus_lower'].isin(bac_genera)].copy()
piv['present'] = (piv['n_genomes_with_ko'] > 0).astype(int)

matrix = piv.pivot_table(index='genus_lower', columns='ko', values='present', fill_value=0)

# Keep only KOs in our gene list that also appear in the matrix
keep_kos = [k for k in all_kos if k in matrix.columns]
matrix = matrix[keep_kos]
print(f'Presence matrix: {matrix.shape}')

bac = bac_base.set_index('genus_lower')
common_genera = sorted(set(matrix.index) & set(bac.index))
matrix      = matrix.loc[common_genera]
bac_aligned = bac.loc[common_genera]
print(f'Common genera (bac_base ∩ presence matrix): {len(common_genera)}')

ko_order  = list(matrix.columns)
ko_to_idx = {ko: i for i, ko in enumerate(ko_order)}
P           = matrix.values.astype(float)              # (n_genera × n_kos)
genome_mb_v = bac_aligned['mean_genome_mb'].values
B_std_v     = bac_aligned['mean_levins_B_std'].values
genus_list  = list(common_genera)
print(f'P matrix: {P.shape}')


Presence matrix: (1574, 343)
Common genera (bac_base ∩ presence matrix): 1574
P matrix: (1574, 343)


## Block 3 — Coreness decile lookup and stratified sampler

In [5]:
# Decile for each KO in the presence matrix (by ko_order)
gdf_idx = gene_df.set_index('KO')
all_deciles = np.array([
    int(gdf_idx.loc[ko, 'coreness_decile']) if ko in gdf_idx.index else 5
    for ko in ko_order
], dtype=int)

# Observed coreness decile distributions for the actual categories
resist_idx_in_matrix  = [ko_to_idx[k] for k in resist_kos   if k in ko_to_idx]
cofactor_idx_in_matrix = [ko_to_idx[k] for k in cofactor_kos if k in ko_to_idx]
resist_hist   = np.bincount(all_deciles[resist_idx_in_matrix],  minlength=10)
cofactor_hist = np.bincount(all_deciles[cofactor_idx_in_matrix], minlength=10)

def stratified_sample(n_target, target_hist, avail_mask, rng_):
    """Sample n_target indices from avail_mask, stratified by coreness decile."""
    selected = []
    avail_idx = np.where(avail_mask)[0]
    for d in range(10):
        pool   = avail_idx[all_deciles[avail_idx] == d]
        n_draw = min(int(target_hist[d]), len(pool))
        if n_draw > 0:
            selected.extend(rng_.choice(pool, size=n_draw, replace=False).tolist())
    # Fill remainder uniformly if strata were exhausted
    n_rem = n_target - len(selected)
    if n_rem > 0:
        already = set(selected)
        pool2 = [i for i in avail_idx if i not in already]
        n_draw2 = min(n_rem, len(pool2))
        if n_draw2 > 0:
            selected.extend(rng_.choice(pool2, size=n_draw2, replace=False).tolist())
    return np.array(selected[:n_target])

print(f'Target sizes: n_A={len(resist_kos)} (resistance), n_B={len(cofactor_kos)} (cofactor)')
print(f'KOs in matrix — resistance: {len(resist_idx_in_matrix)}, cofactor: {len(cofactor_idx_in_matrix)}')

Target sizes: n_A=106 (resistance), n_B=7 (cofactor)
KOs in matrix — resistance: 75, cofactor: 4


## Block 4 — Observed Δβ from NB03 category results

In [6]:
cat_res = pd.read_csv(DATA / '03_category_pgls_results.csv')
beta_R = float(cat_res.loc[cat_res['label'] == 'F1.1_resistance', 'beta'].values[0])
beta_C = float(cat_res.loc[cat_res['label'] == 'F1.4_cofactor',  'beta'].values[0])
delta_beta_obs = beta_R - beta_C
print(f'beta_resistance = {beta_R:.5f}')
print(f'beta_cofactor   = {beta_C:.5f}')
print(f'Observed Δβ     = {delta_beta_obs:.5f}')

beta_resistance = 0.00252
beta_cofactor   = -0.03274
Observed Δβ     = 0.03526


## Block 5 — Run 1,000 permutations

In [7]:
def density_beta(ko_indices):
    """Compute per-genus density for ko_indices and return PGLS beta (or None if degenerate)."""
    n_ko    = P[:, ko_indices].sum(axis=1)
    density = n_ko / genome_mb_v
    if density.std() < 1e-10:
        return None
    density_z = (density - density.mean()) / density.std()
    df_tmp = pd.DataFrame({
        'genus_lower':       genus_list,
        'density_z':         density_z,
        'mean_levins_B_std': B_std_v,
    })
    try:
        res = run_pgls(df_tmp, TREE_PATH, response='mean_levins_B_std',
                       predictors=['density_z'])   # taxon_col defaults to 'genus_lower'
        return res.get('beta', None)
    except Exception:
        return None

rng      = np.random.default_rng(SEED)
null_dbs = []

print(f'Running {N_PERM} permutations...')
for i in range(N_PERM):
    if i % 200 == 0:
        print(f'  {i}/{N_PERM}')
    avail = np.ones(len(ko_order), dtype=bool)
    idx_A = stratified_sample(len(resist_kos),   resist_hist,   avail.copy(), rng)
    avail[idx_A] = False
    idx_B = stratified_sample(len(cofactor_kos), cofactor_hist, avail, rng)
    bA = density_beta(idx_A)
    bB = density_beta(idx_B)
    if bA is not None and bB is not None:
        null_dbs.append(bA - bB)

null_dbs = np.array(null_dbs)
if len(null_dbs) == 0:
    raise RuntimeError('All permutations failed — check TREE_PATH and column names.')

emp_p    = (null_dbs >= delta_beta_obs).mean()
print(f'\nValid iterations: {len(null_dbs)}')
print(f'Null: median={np.median(null_dbs):.4f}, SD={null_dbs.std():.4f}, '
      f'range=[{null_dbs.min():.4f}, {null_dbs.max():.4f}]')
print(f'Observed Δβ = {delta_beta_obs:.4f}, empirical p = {emp_p:.3f}')


Running 1000 permutations...
  0/1000


/home/hmacgregor/.local/lib/python3.13/site-packages/dendropy/datamodel/treemodel/_tree.py:1510: UserWarning: Calculating MRCA on an unrooted tree implicitly implicitly treats seed node as root. Set tree.is_rooted = True to silence this warning.
  warnings.warn(


/home/hmacgregor/.local/lib/python3.13/site-packages/dendropy/datamodel/treemodel/_tree.py:1510: UserWarning: Calculating MRCA on an unrooted tree implicitly implicitly treats seed node as root. Set tree.is_rooted = True to silence this warning.
  warnings.warn(


/home/hmacgregor/.local/lib/python3.13/site-packages/dendropy/datamodel/treemodel/_tree.py:1510: UserWarning: Calculating MRCA on an unrooted tree implicitly implicitly treats seed node as root. Set tree.is_rooted = True to silence this warning.
  warnings.warn(


/home/hmacgregor/.local/lib/python3.13/site-packages/dendropy/datamodel/treemodel/_tree.py:1510: UserWarning: Calculating MRCA on an unrooted tree implicitly implicitly treats seed node as root. Set tree.is_rooted = True to silence this warning.
  warnings.warn(


...

/home/hmacgregor/.local/lib/python3.13/site-packages/dendropy/datamodel/treemodel/_tree.py:1510: UserWarning: Calculating MRCA on an unrooted tree implicitly implicitly treats seed node as root. Set tree.is_rooted = True to silence this warning.
  warnings.warn(


/home/hmacgregor/.local/lib/python3.13/site-packages/dendropy/datamodel/treemodel/_tree.py:1510: UserWarning: Calculating MRCA on an unrooted tree implicitly implicitly treats seed node as root. Set tree.is_rooted = True to silence this warning.
  warnings.warn(


/home/hmacgregor/.local/lib/python3.13/site-packages/dendropy/datamodel/treemodel/_tree.py:1510: UserWarning: Calculating MRCA on an unrooted tree implicitly implicitly treats seed node as root. Set tree.is_rooted = True to silence this warning.
  warnings.warn(


/home/hmacgregor/.local/lib/python3.13/site-packages/dendropy/datamodel/treemodel/_tree.py:1510: UserWarning: Calculating MRCA on an unrooted tree implicitly implicitly treats seed node as root. Set tree.is_rooted = True to silence this warning.
  warnings.warn(



Valid iterations: 1000
Null: median=-0.0062, SD=0.0069, range=[-0.0286, 0.0183]
Observed Δβ = 0.0353, empirical p = 0.000


## Block 6 — Comparison families (AMR, TCS, ABC) from NB19

For each family, compute observed within-family Δβ (max − min subcategory β).
Running the coreness-matched null for comparison families requires a separate
Spark query (same schema, different KO lists) — add a cache block analogous to
Block 1 once the AMR/TCS/ABC KO lists from NB19 inputs are available.

In [8]:
int_res = pd.read_csv(DATA / 'internal_structure_results.csv')
print(int_res[['parent_category','subcategory','n_kos','beta','p_raw']].to_string())

family_rows = []
for fam, grp in int_res.groupby('parent_category'):
    obs_db = float(grp['beta'].max() - grp['beta'].min())
    n_hi   = int(grp.loc[grp['beta'].idxmax(), 'n_kos'])
    n_lo   = int(grp.loc[grp['beta'].idxmin(), 'n_kos'])
    family_rows.append({'family': fam, 'n_kos_hi': n_hi, 'n_kos_lo': n_lo,
                        'observed_delta_beta': obs_db, 'emp_p': float('nan')})
    print(f'{fam}: observed Δβ = {obs_db:.4f}  (n_hi={n_hi}, n_lo={n_lo})')

# TODO: add per-family presence matrix cache blocks and permutation loops here.
# Pattern is identical to Blocks 1–5; replace all_kos with family-specific KO list,
# P with family-specific matrix, and n_A/n_B with family subcategory sizes.

          parent_category                     subcategory  n_kos      beta     p_raw
0        ABC transporters                       Lipid/LPS     16 -0.029518  0.000005
1        ABC transporters                Vitamin/cofactor     10 -0.014816  0.071792
2        ABC transporters              Amino acid/peptide     87 -0.008217  0.144117
3        ABC transporters       Inorganic ion (non-metal)     35 -0.003175  0.513334
4        ABC transporters                  Drug/multidrug     23 -0.000425  0.944062
5        ABC transporters              Sugar/carbohydrate     56  0.002068  0.684269
6        ABC transporters              Other/unclassified    248  0.002909  0.647283
7                     AMR          Enzymatic inactivation    106  0.005827  0.209085
8                     AMR  Target modification/protection     86  0.008626  0.129243
9                     AMR                    Efflux pumps     49  0.018015  0.005334
10  Two-component systems           Phosphotransfer/other    288 

## Block 7 — Figure

In [9]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

BLUE, RED, ORANGE = '#0072B2', '#CC3311', '#E69F00'
GREEN, CYAN       = '#009E73', '#56B4E9'

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Panel A — metal null distribution
ax = axes[0]
ax.hist(null_dbs, bins=50, color=BLUE, alpha=0.75, edgecolor='none',
        label=f'Null Δβ ({len(null_dbs)} permutations)')
ax.axvline(delta_beta_obs, color=RED, lw=2.5, ls='--',
           label=f'Observed Δβ = {delta_beta_obs:.3f}\nemp_p = {emp_p:.3f}')
ax.set_xlabel('Δβ (β$_{large\ group}$ − β$_{small\ group}$)', fontsize=11)
ax.set_ylabel('Count', fontsize=11)
ax.set_title('Metal gene set\n(n=106 resistance vs n=7 cofactor)', fontsize=11, fontweight='bold')
ax.legend(fontsize=9)

# Panel B — comparison families (observed Δβ only)
ax = axes[1]
fams    = [d['family'] for d in family_rows]
obs_dbs = [d['observed_delta_beta'] for d in family_rows]
cols    = [ORANGE, GREEN, CYAN][:len(fams)]
ax.barh(fams, obs_dbs, color=cols, alpha=0.8)
ax.axvline(delta_beta_obs, color=RED, lw=1.5, ls='--',
           label=f'Metal observed Δβ = {delta_beta_obs:.3f}')
ax.set_xlabel('Observed within-family Δβ', fontsize=11)
ax.set_title('Comparison families\n(null distributions TBD per family)', fontsize=11)
ax.legend(fontsize=9)

plt.tight_layout()
fig.savefig(FIGS / 'split_magnitude_permutation.png', dpi=150, bbox_inches='tight')
plt.close()
print('Figure saved: figures/split_magnitude_permutation.png')

<>:16: SyntaxWarning: invalid escape sequence '\ '
<>:16: SyntaxWarning: invalid escape sequence '\ '
/tmp/ipykernel_6985/321843139.py:16: SyntaxWarning: invalid escape sequence '\ '
  ax.set_xlabel('Δβ (β$_{large\ group}$ − β$_{small\ group}$)', fontsize=11)


Figure saved: figures/split_magnitude_permutation.png


## Block 8 — Save results

In [10]:
out = [{
    'family':              'Metal gene set',
    'n_kos_A':             len(resist_kos),
    'n_kos_B':             len(cofactor_kos),
    'observed_delta_beta': float(delta_beta_obs),
    'null_delta_beta_median': float(np.median(null_dbs)),
    'null_delta_beta_sd':     float(null_dbs.std()),
    'n_valid_perms':       int(len(null_dbs)),
    'emp_p':               float(emp_p),
}] + family_rows

out_df = pd.DataFrame(out)
out_df.to_csv(DATA / 'split_magnitude_permutation.csv', index=False)
print(out_df.to_string())
print('\nSaved: data/split_magnitude_permutation.csv')
print('Saved: figures/split_magnitude_permutation.png')

                  family  n_kos_A  n_kos_B  observed_delta_beta  null_delta_beta_median  null_delta_beta_sd  n_valid_perms  emp_p  n_kos_hi  n_kos_lo
0         Metal gene set    106.0      7.0             0.035259               -0.006202            0.006906         1000.0    0.0       NaN       NaN
1       ABC transporters      NaN      NaN             0.032427                     NaN                 NaN            NaN    NaN     248.0      16.0
2                    AMR      NaN      NaN             0.012187                     NaN                 NaN            NaN    NaN      49.0     106.0
3  Two-component systems      NaN      NaN             0.006162                     NaN                 NaN            NaN    NaN     112.0     288.0

Saved: data/split_magnitude_permutation.csv
Saved: figures/split_magnitude_permutation.png
